# Multi-Class Land Cover Classification - Grid Search

This notebook performs hyperparameter tuning for a 4-class land cover classification on the campus:
- 0: Water
- 1: Forest
- 2: Soil
- 3: Buildings

It uses 10 spectral features and performs a 5-fold GridSearchCV.

In [ ]:
import os
import ee
from dotenv import load_dotenv
import pandas as pd
import numpy as np
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

load_dotenv()
ee.Initialize(project=os.getenv('EE_PROJECT_ID'))
print("✅ Earth Engine initialized.")

In [ ]:
campus_geojson = {
    "type": "Polygon",
    "coordinates": [
        [
            [80.01710357666015, 23.173962177472703],
            [80.03259601593017, 23.165361215115187],
            [80.03654422760009, 23.172502420044232],
            [80.026802444458,   23.181694681000845],
            [80.01542987823485, 23.176960548201308],
        ]
    ]
}
campus = ee.Geometry(campus_geojson)

def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud  = 1 << 10
    cirrus = 1 << 11
    mask = qa.bitwiseAnd(cloud).eq(0).And(qa.bitwiseAnd(cirrus).eq(0))
    return image.updateMask(mask).divide(10000)

print("Loading Earth Engine data...")
dataset = (
    ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterDate('2026-01-01', '2026-02-28')
      .filterBounds(campus)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
      .map(mask_s2_clouds)
)
composite = dataset.median().clip(campus)
print("✅ Sentinel-2 median composite created.")

In [ ]:
ndvi = composite.normalizedDifference(['B8', 'B4']).rename('NDVI')
ndwi = composite.normalizedDifference(['B3', 'B8']).rename('NDWI')
ndbi = composite.normalizedDifference(['B11', 'B8']).rename('NDBI')

savi = composite.expression(
    '1.5 * (NIR - RED) / (NIR + RED + 0.5)',
    {'NIR': composite.select('B8'), 'RED': composite.select('B4')}
).rename('SAVI')

composite = composite.addBands([ndvi, ndwi, ndbi, savi])

bands = ['B2', 'B3', 'B4', 'B8', 'B11', 'B12', 'NDVI', 'NDWI', 'NDBI', 'SAVI']
print(f"Feature bands: {bands}")

In [ ]:
water     = ee.FeatureCollection('users/cosypix/multi_water_points')
forest    = ee.FeatureCollection('users/cosypix/multi_forest_points')
soil      = ee.FeatureCollection('users/cosypix/multi_soil_points')
buildings = ee.FeatureCollection('users/cosypix/multi_buildings_points')

all_points = water.merge(forest).merge(soil).merge(buildings)

print("Sampling feature regions...")
training = (
    composite.select(bands)
             .sampleRegions(
                 collection=all_points,
                 properties=['label'],
                 scale=10
             )
             .filter(ee.Filter.notNull(bands + ['label']))
)

data_list = training.getInfo()['features']
print(f"✅ Extraction complete! {len(data_list)} points found.")

In [ ]:
df = pd.DataFrame([feat['properties'] for feat in data_list])
X  = df[bands]
y  = df['label']

print(f"Dataset shape: {X.shape}")
print(f"Class distribution:\n{y.value_counts().sort_index()}")

In [ ]:
rf_grid = {
    'n_estimators':     [50, 100, 200],
    'min_samples_leaf': [1, 3, 5],
    'max_samples':      [0.3, 0.5, 0.7, None],
    'max_depth':        [10, 20, None]
}

svm_grid = [
    {'kernel': ['rbf'], 'gamma': [0.01, 0.1, 0.5, 1.0], 'C': [1, 10, 100]},
    {'kernel': ['linear'], 'gamma': ['scale'], 'C': [1, 10, 100]}
]

xgb_grid = {
    'n_estimators':  [50, 100, 200],
    'learning_rate': [0.01, 0.05, 0.1, 0.3],
    'max_depth':     [3, 5, 10],
    'num_class':     [4]
}

pipelines = [
    ('RF',  RandomForestClassifier(random_state=42), rf_grid),
    ('SVM', SVC(random_state=42), svm_grid),
    ('XGB', XGBClassifier(random_state=42, objective='multi:softmax', eval_metric='mlogloss'), xgb_grid)
]

In [ ]:
all_results = []
best_per_model = {}

for name, model, grid in pipelines:
    print(f"\n🔍 Running GridSearchCV for {name}...")
    search = GridSearchCV(model, grid, cv=5, scoring='accuracy', return_train_score=True, n_jobs=-1)
    search.fit(X, y)
    
    cv_res = pd.DataFrame(search.cv_results_)
    cv_res['Model'] = name
    cv_res['params_str'] = cv_res['params'].astype(str)
    
    best_per_model[name] = {
        'best_params': search.best_params_,
        'best_score':  search.best_score_
    }
    print(f"   ✅ Done. Best CV accuracy: {search.best_score_:.4f}")

In [ ]:
print("=" * 55)
print("       BEST HYPERPARAMETERS PER MODEL")
print("=" * 55)
for model_name, info in best_per_model.items():
    print(f"\n🏆 {model_name}")
    print(f"   Best CV Accuracy : {info['best_score']:.4f}")
    print(f"   Best Parameters  :")
    for k, v in info['best_params'].items():
        print(f"      {k:20s} = {v}")
